# 103 — Búsqueda híbrida y fusión de rankings

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** Con `k = 60`:
`RRF(P) = 1/61 + 1/63 = 0.0164 + 0.0159 = 0.0323`;
`RRF(Q) = 1/62 + 1/61 = 0.0161 + 0.0164 = 0.0325`;
`RRF(R) = 1/63 = 0.0159`; `RRF(S) = 1/62 = 0.0161`.
Ranking: **Q > P > S > R**. Q gana por consenso (2.º y 1.º).

**Ejercicio 2.** Con `k = 0`: `P = 1/1 + 1/3 = 1.333`, `Q = 1/2 + 1/1 = 1.5`,
`R = 1/3 = 0.333`, `S = 1/2 = 0.5` → sigue Q > P, pero los pesos de los primeros
puestos se disparan (un 1.º aislado vale 1.0, casi el doble que un 2.º + 3.º = 0.83).
Con `k = 1000` todos los términos rondan 1/1000: las diferencias de puesto casi
desaparecen y domina simplemente *en cuántas ramas apareces* (P y Q con 2 apariciones
≈ 0.002; R y S con 1 ≈ 0.001). `k` interpola entre "mandan los primeros puestos" y
"manda el número de apariciones".

**Ejercicio 3.** Min-max BM25 (min 3.1, max 12.4): P = 1.0, Q = (11.9−3.1)/9.3 ≈ 0.946,
R = 0. Min-max denso (min 0.85, max 0.91): Q = 1.0, S = 0.5, P = 0. Combinación α = 0.5
(ausente = 0): P = 0.5, Q ≈ 0.973, R = 0, S = 0.25 → **Q > P > S > R**. Mismo orden que
RRF aquí, pero P es el sensible: su score depende por completo de que 0.85 sea el mínimo
del top-3 denso — si el denso devolviera un 4.º documento con 0.80, P saltaría de 0 a
0.42 sin que nada relevante cambiara. RRF no sufre ese artefacto porque ignora las magnitudes.

**Ejercicio 4.** Contrato estable (`kind`, `evidence`); valores dependientes de la semilla.


In [ ]:
result = run_lab("retrieval", seed=103)
assert result["kind"] == "retrieval"
assert result["evidence"]
show(result)


In [ ]:
from collections import defaultdict

def rrf(rankings, k=60):
    s = defaultdict(float)
    for r in rankings:
        for puesto, doc in enumerate(r, start=1):
            s[doc] += 1 / (k + puesto)
    return dict(sorted(s.items(), key=lambda kv: -kv[1]))

R1 = ["P", "Q", "R"]; R2 = ["Q", "S", "P"]
for k in (60, 0, 1000):
    print(f"k={k}:", {d: round(v, 4) for d, v in rrf([R1, R2], k=k).items()})

# Ejercicio 3: score fusion min-max + α=0.5
bm25 = {"P": 12.4, "Q": 11.9, "R": 3.1}
denso = {"Q": 0.91, "S": 0.88, "P": 0.85}
def minmax(d):
    lo, hi = min(d.values()), max(d.values())
    return {k: (v - lo) / (hi - lo) for k, v in d.items()}
nb, nd = minmax(bm25), minmax(denso)
docs = set(bm25) | set(denso)
comb = {d: 0.5 * nb.get(d, 0) + 0.5 * nd.get(d, 0) for d in docs}
print("score fusion:", dict(sorted(comb.items(), key=lambda kv: -kv[1])))


## Reflexión

1. En el ejemplo, A gana sin ser primero en ningún ranking. ¿Qué propiedad de RRF produce eso y por qué suele ser deseable en recuperación?
2. ¿Qué pasaría con RRF si usaras k = 0? Calcula RRF(C) y RRF(A) del ejemplo con k = 0 y comprueba si el ganador cambia y por qué.
3. ¿En qué situación una fusión híbrida rendiría peor que la mejor rama sola, y qué experimento lo detectaría antes de desplegarla?
